In [1]:
import io
import pandas as pd
import re

import subprocess

In [2]:
def parse_chain_string(chain_data_string):
    
    # Find all full chain entries using a lookahead for 'id:[' and greedily grab until the next 'id:[' or end of string
    chain_entries = re.findall(r'(id:\[.*?)(?=id:\[|$)', chain_data_string, re.DOTALL)

    parsed_chains_list = []

    for entry in chain_entries:
        chain_info = {}
        matches = re.findall(r'([^:]+):\[(.*?)\]', entry, re.DOTALL)
        for key, value in matches:
            chain_info[key] = value
        parsed_chains_list.append(chain_info)

    return parsed_chains_list

# Clean PDA Info

In [3]:

data = pd.read_csv('pda_exported_designs.csv')

training_data = pd.read_csv('../../data/training_data.csv').query('type == 1').copy()
training_data['pdb_id'] = training_data['name'].str.extract(r'([0-9][A-Za-z0-9]{3})')
common_pdb_ids = set(data['pdb_code'].str.lower()).intersection(set(training_data['pdb_id'].str.lower()))
cleaned_data = data[~data['pdb_code'].str.lower().isin(common_pdb_ids)].reset_index(drop=True)

sequ_info = cleaned_data[['pdb_code', 'chains','release_date']].copy()
sequ_info['parsed_chains'] = sequ_info['chains'].apply(parse_chain_string)

all_chains_records = []

for index, row in sequ_info.iterrows():
    pdb_code = row['pdb_code'] 
    release_date = row['release_date']
    parsed_chains_for_row = row['parsed_chains']

    if parsed_chains_for_row:
        for chain_dict in parsed_chains_for_row:
            chain_record = {
                'pdb_code': pdb_code,
                'release_date': release_date,
                **chain_dict
            }
            all_chains_records.append(chain_record)
    else:
        all_chains_records.append({
            'pdb_code': pdb_code,
        })

chains_df = pd.DataFrame(all_chains_records)

# Filter PDA Info

In [4]:
denovo_sequences_df = chains_df[
    chains_df['chain_source'].isin(['Synthetic','synthetic construct', 'unknown', 'artificial gene'])
].copy() 


if 'sequence' in denovo_sequences_df.columns:
    denovo_sequences_df['sequence'] = denovo_sequences_df['sequence'].fillna('')

# Filter duplicates
denovo_sequences_df['sequence_length'] = denovo_sequences_df['sequence'].apply(len)
denovo_sequences_df = denovo_sequences_df.drop_duplicates(subset='sequence').reset_index(drop=True)
# print(len(denovo_sequences_df))


# Filter sequences out of range
denovo_sequences_df = denovo_sequences_df[
    (denovo_sequences_df['sequence_length'] >= 40) & 
    (denovo_sequences_df['sequence_length'] <= 250)
].reset_index(drop=True)

print(f"{len(denovo_sequences_df)} sequences of specified length")

# Filter sequences containing parentheses
condition_contains_open_paren = denovo_sequences_df['sequence'].str.contains(r'\(', regex=True)
condition_contains_close_paren = denovo_sequences_df['sequence'].str.contains(r'\)', regex=True)
rows_to_filter_out = (
    condition_contains_open_paren |
    condition_contains_close_paren
)
denovo_sequences_df = denovo_sequences_df[~rows_to_filter_out].reset_index(drop=True)
denovo_sequences_df = denovo_sequences_df.dropna().reset_index(drop=True)

print(f"{len(denovo_sequences_df)} sequences without parentheses")


755 sequences of specified length
537 sequences without parentheses


# Download and clean PDBs


In [ ]:
for _, row in denovo_sequences_df.iterrows():
    pdb_code = row['pdb_code']
    chain_id = row['id'][0]
    
    # Command needs Rosetta installed
    command = f"python2.7 $ROSETTA/tools/protein_tools/scripts/clean_pdb.py {pdb_code} {chain_id}"
    # command = f"echo 'pdb={pdb_code} chain={chain_id}'"

    try:
        subprocess.run(command, shell=True, check=True)
        print(f"Processed: {pdb_code} with chain {chain_id}")
    except subprocess.CalledProcessError as e:
        print(f"Failed to process {pdb_code} with chain {chain_id}: {e}")

Relax downloaded structures on cluster:

In [ ]:
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --mem=8G
#SBATCH --time=0-30:00:00
#SBATCH --partition=paul
#SBATCH --output=log/log.%J
#SBATCH --job-name="relax"

job=$SLURM_ARRAY_JOB_ID
task=$SLURM_ARRAY_TASK_ID
ROSETTA="/software/all/Rosetta/3.14-gompi-2023b"

module load "Rosetta"

$ROSETTA/bin/relax.mpi.linuxgccrelease -s $1 -nstruct 6 -in:file:fullatom -relax:constrain_relax_to_start_coords -relax:ramp_constraints false -out:prefix relax_

# Add Relax Info

In [5]:
with open('relax_score.sc', 'r') as file:
    lines = file.readlines()

filtered_lines = [line.strip() for line in lines if line.strip() and not line.startswith('SEQUENCE:')]

header_line = filtered_lines[0].replace('SCORE:', '').strip()
column_names = [col.strip() for col in header_line.split(' ') if col.strip()]

data_lines = [line.replace('SCORE:', '').strip() for line in filtered_lines[1:] if line.startswith('SCORE: ')]

relax = pd.read_csv(io.StringIO('\n'.join(data_lines)), sep='\s+', header=None, names=column_names)
relax['pdb_code'] = relax['description'].str.split('_').str[1].str.lower()
relax['chain_id'] = relax['description'].str.split('_').str[2]

filtered_relax = relax.loc[relax.groupby(['pdb_code', 'chain_id'])['total_score'].idxmin()].reset_index(drop=True)
filtered_relax['prim_key'] = filtered_relax['pdb_code'] + '_' + filtered_relax['chain_id']
denovo_sequences_df['prim_key'] = denovo_sequences_df['pdb_code'].str.lower() + '_' + denovo_sequences_df['id'].str[0]


merged_relax = pd.merge(denovo_sequences_df, filtered_relax, 
                     left_on=['prim_key'], right_on=['prim_key'], how='right')

# Add Predictions

Predictions were generated like it is described on github.

In [6]:
predictions = pd.read_csv('pred_pda.csv',names=['filename', 'probability'])
predictions['pdb_code'] = predictions['filename'].str.split('_').str[1].str.lower()
predictions['chain_id'] = predictions['filename'].str.split('_').str[2]
predictions['prim_key'] = predictions['pdb_code'] + '_' + predictions['chain_id']
# Add sequence information to the predictions
predictions_merged = pd.merge(predictions, merged_relax, left_on='prim_key', right_on='prim_key', how='inner')
predictions_merged['relative_score'] = predictions_merged['total_score'] / predictions_merged['sequence_length']
predictions_merged = predictions_merged[predictions_merged['relative_score'] < -0.50]


# Analyse
Search for wrong predicted *de novo* proteins

In [7]:
# write fasta file with sequences
fasta_string = predictions_merged.apply(
    lambda row: f">{row['prim_key']}\n{row['sequence']}\n", axis=1
).str.cat(sep='')
with open('denovo.fasta', 'w') as fasta_file:
    fasta_file.write(fasta_string)

# Blast
Use the fasta to Blast search and the script extract_json_to_csv.py to convert the blastout json to extracted_blast_results.csv

In [8]:
blast_data = pd.read_csv('extracted_blast_results.csv')

In [9]:
blast_data = blast_data[blast_data['percent_id'] < 100].reset_index(drop=True)
filter = blast_data[blast_data['sciname'] != "synthetic construct"].reset_index(drop=True)

filtered_denovo = predictions_merged[~predictions_merged['prim_key'].isin(filter['query_title'])]
wrong_classified = filtered_denovo[(filtered_denovo['probability'].astype(float) < 0.5)]

print(f"De novo proteins after filtering: {len(filtered_denovo)}")
print(f"Missed proteins: {len(wrong_classified)}")
print(f"Percent correct: {(1-(len(wrong_classified) / len(filtered_denovo))) * 100:.2f}%")

De novo proteins after filtering: 186
Missed proteins: 73
Percent correct: 60.75%
